In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import RobustScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# ==================== КОНФИГУРАЦИЯ ====================
SOURCE_ROOT = "final_data"
TARGET_ROOT = "final_data_encoded"
os.makedirs(TARGET_ROOT, exist_ok=True)

# ==================== ИНФОРМАЦИЯ О ДАТАСЕТАХ ====================
DATASET_INFO = {
    "Credit Card Fraud": {
        "path": os.path.join(SOURCE_ROOT, "creditcard.csv"),
        "target": "Class",
        "type": "tabular"
    },
    "IEEE-CIS": {
        "path": os.path.join(SOURCE_ROOT, "ieee_train_transaction.csv"),
        "target": "isFraud",
        "type": "tabular"
    },
    "PaySim": {
        "path": os.path.join(SOURCE_ROOT, "paysim.csv"),
        "target": "isFraud",
        "type": "tabular"
    },
    "AIZBank": {
        "path": os.path.join(SOURCE_ROOT, "aizbank.csv"),
        "target": "label",
        "type": "tabular"
    },
    "Elliptic (Bitcoin)": {
        "path": os.path.join(SOURCE_ROOT, "Elliptic", "elliptic_txs_features.csv"),
        "label_path": os.path.join(SOURCE_ROOT, "Elliptic", "elliptic_txs_classes.csv"),
        "type": "graph"
    },
    "Amazon Reviews": {
        "path": os.path.join(SOURCE_ROOT, "Amazon", "Amazon_full.csv"),
        "target": "label",
        "type": "graph"
    },
    "Yelp Reviews": {
        "path": os.path.join(SOURCE_ROOT, "YelpChi", "YelpChi_full.csv"),
        "target": "label",
        "type": "graph"
    },
    "Click Fraud": {
        "path": os.path.join(SOURCE_ROOT, "click_fraud_sample.csv"),
        "target": "is_attributed",
        "type": "temporal"
    },
    "Social Honeypot": {
        "path": os.path.join(SOURCE_ROOT, "social_honeypot.csv"),
        "target": "label",
        "type": "tabular"
    },
    "Cresci-2017": {
        "path": os.path.join(SOURCE_ROOT, "cresci_merged.csv"),
        "target": "label",
        "type": "tabular"
    },
    "Midterm-2018": {
        "path": os.path.join(SOURCE_ROOT, "midterm_merged.csv"),
        "target": "label",
        "type": "tabular"
    },
    "MGTAB": {
        "path": os.path.join(SOURCE_ROOT, "MGTAB", "features.csv"),
        "label_path": os.path.join(SOURCE_ROOT, "MGTAB", "labels_bot.csv"),
        "type": "graph"
    },
}

# ==================== ФУНКЦИЯ ПРЕДОБРАБОТКИ ====================
def preprocess_dataframe(df, target_col, cat_threshold=10):
    """
    Принимает DataFrame с признаками и целевой колонкой.
    Возвращает X (numpy), y (numpy) после кодирования, масштабирования, винсоризации.
    """
    # Целевая переменная
    y = df[target_col].copy()
    if y.dtype == 'object':
        y = y.map({'yes': 1, 'no': 0, 'bot': 1, 'human': 0}).fillna(0).astype(int)
    else:
        y = y.fillna(0).astype(int)

    X = df.drop(columns=[target_col])

    # === Кодирование категориальных признаков ===
    categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    for col in categorical_cols:
        uniq = X[col].nunique()
        if uniq <= cat_threshold:
            # One-hot
            ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
            ohe_data = ohe.fit_transform(X[[col]].astype(str))
            ohe_cols = [f"{col}_{cat}" for cat in ohe.categories_[0]]
            ohe_df = pd.DataFrame(ohe_data, columns=ohe_cols, index=X.index)
            X = pd.concat([X, ohe_df], axis=1)
        else:
            # Label encoding
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
    X = X.drop(columns=categorical_cols, errors='ignore')

    # Приводим все колонки к числовому типу
    X = X.apply(pd.to_numeric, errors='coerce')

    # === Удаление признаков с долей пропусков > 80% ===
    missing_ratio = X.isnull().mean()
    high_miss = missing_ratio[missing_ratio > 0.8].index
    X = X.drop(columns=high_miss)

    # === Заполнение пропусков медианой ===
    imputer = SimpleImputer(strategy='median')
    X = imputer.fit_transform(X)

    # === Винсоризация (1-99 процентили) ===
    for i in range(X.shape[1]):
        lo = np.percentile(X[:, i], 1)
        hi = np.percentile(X[:, i], 99)
        X[:, i] = np.clip(X[:, i], lo, hi)

    # === Масштабирование ===
    scaler = RobustScaler()
    X = scaler.fit_transform(X)

    return X, y.values

# ==================== ОБРАБОТКА ВСЕХ ДАТАСЕТОВ ====================
for name, info in DATASET_INFO.items():
    print(f"Обработка {name}...")
    try:
        if info['type'] in ('tabular', 'temporal'):
            df = pd.read_csv(info['path'])
            if info['type'] == 'temporal' and 'click_time' in df.columns:
                df = df.drop(columns=['click_time'])  # удаляем временную метку
            X, y = preprocess_dataframe(df, info['target'])
            # Сохраняем
            np.save(os.path.join(TARGET_ROOT, f"{name.replace(' ', '_')}_X.npy"), X)
            np.save(os.path.join(TARGET_ROOT, f"{name.replace(' ', '_')}_y.npy"), y)
            print(f"  Сохранено: X shape {X.shape}, y shape {y.shape}")

        elif info['type'] == 'graph':
            if name == "Elliptic (Bitcoin)":
                df_feat = pd.read_csv(info['path'], header=None)
                df_feat.columns = ['txId', 'time_step'] + [f'f{i}' for i in range(df_feat.shape[1]-2)]
                df_class = pd.read_csv(info['label_path'])
                df_class['label'] = df_class['class'].map({'1': 0, '2': 1, 'unknown': -1})
                df_class = df_class[df_class['label'] != -1]
                df = df_feat.merge(df_class, on='txId')
                df = df.drop(columns=['txId', 'time_step', 'class'])
                X, y = preprocess_dataframe(df, 'label')
            elif name == "Amazon Reviews":
                df = pd.read_csv(info['path'], header=None, skiprows=1)
                df.columns = [f'feat_{i}' for i in range(df.shape[1]-1)] + ['label']
                X, y = preprocess_dataframe(df, 'label')
            elif name == "Yelp Reviews":
                df = pd.read_csv(info['path'])
                X, y = preprocess_dataframe(df, 'label')
            elif name == "MGTAB":
                df_feat = pd.read_csv(info['path'], header=None, low_memory=False)
                df_labels = pd.read_csv(info['label_path'], header=None)
                y = df_labels[0].values
                # Обрезаем признаки до количества меток
                if len(df_feat) > len(y):
                    df_feat = df_feat.iloc[:len(y)]
                X = df_feat.values.astype(np.float32)
                # Для MGTAB пропусков нет, но на всякий случай
                imputer = SimpleImputer(strategy='median')
                X = imputer.fit_transform(X)
                # Винсоризация
                for i in range(X.shape[1]):
                    lo = np.percentile(X[:, i], 1)
                    hi = np.percentile(X[:, i], 99)
                    X[:, i] = np.clip(X[:, i], lo, hi)
                # Масштабирование
                scaler = RobustScaler()
                X = scaler.fit_transform(X)
            else:
                continue
            np.save(os.path.join(TARGET_ROOT, f"{name.replace(' ', '_')}_X.npy"), X)
            np.save(os.path.join(TARGET_ROOT, f"{name.replace(' ', '_')}_y.npy"), y)
            print(f"  Сохранено: X shape {X.shape}, y shape {y.shape}")

    except Exception as e:
        print(f"  Ошибка при обработке {name}: {e}")

print("\nПредобработка завершена. Данные сохранены в", TARGET_ROOT)

Обработка Credit Card Fraud...
  Сохранено: X shape (284807, 30), y shape (284807,)
Обработка IEEE-CIS...
  Сохранено: X shape (590540, 367), y shape (590540,)
Обработка PaySim...
  Сохранено: X shape (6362620, 12), y shape (6362620,)
Обработка AIZBank...
  Сохранено: X shape (45211, 27), y shape (45211,)
Обработка Elliptic (Bitcoin)...
  Сохранено: X shape (46564, 165), y shape (46564,)
Обработка Amazon Reviews...
  Сохранено: X shape (11944, 25), y shape (11944,)
Обработка Yelp Reviews...
  Сохранено: X shape (45954, 32), y shape (45954,)
Обработка Click Fraud...
  Сохранено: X shape (100000, 5), y shape (100000,)
Обработка Social Honeypot...
  Сохранено: X shape (41499, 6), y shape (41499,)
Обработка Cresci-2017...
  Сохранено: X shape (6105, 12), y shape (6105,)
Обработка Midterm-2018...
  Сохранено: X shape (50538, 21), y shape (50538,)
Обработка MGTAB...
  Сохранено: X shape (10200, 788), y shape (10200,)

Предобработка завершена. Данные сохранены в final_data_encoded


In [5]:
import numpy as np
import pandas as pd
import time
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, auc, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# ==================== ПАРАМЕТРЫ ====================
RANDOM_STATES = [42, 43, 44, 45, 46]
TRAIN_SIZE = 0.7
VAL_SIZE = 0.15
TEST_SIZE = 0.15
TUNE_SUBSAMPLE = 30000   # для подбора гиперпараметров на больших датасетах

# Список всех датасетов (имена совпадают с именами в папке final_data_encoded)
DATASET_NAMES = [
    "Credit Card Fraud", "IEEE-CIS", "PaySim", "AIZBank",
    "Elliptic (Bitcoin)", "Amazon Reviews", "Yelp Reviews",
    "Click Fraud", "Social Honeypot", "Cresci-2017", "Midterm-2018", "MGTAB"
]

# ==================== ФУНКЦИИ ====================
def split_stratified(X, y, train_ratio, val_ratio, test_ratio, random_state):
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=(val_ratio+test_ratio), stratify=y, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=test_ratio/(val_ratio+test_ratio),
        stratify=y_temp, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

def find_best_threshold(y_val, proba_val):
    if hasattr(y_val, 'values'):
        y_val = y_val.values
    prec, rec, thresholds = precision_recall_curve(y_val, proba_val)
    f1_scores = 2 * (prec * rec) / (prec + rec + 1e-8)
    best_idx = np.argmax(f1_scores)
    if best_idx < len(thresholds):
        return thresholds[best_idx]
    return 0.5

def compute_metrics(y_test, proba, best_thr):
    if hasattr(y_test, 'values'):
        y_test = y_test.values
    y_pred = (proba >= best_thr).astype(int)
    f1 = f1_score(y_test, y_pred)
    K = y_test.sum()
    if K > 0:
        top_k_idx = np.argsort(proba)[-K:]
        prec_at_k = y_test[top_k_idx].mean()
    else:
        prec_at_k = np.nan
    prec, rec, _ = precision_recall_curve(y_test, proba)
    auprc = auc(rec, prec)
    auroc = roc_auc_score(y_test, proba)
    return auprc, auroc, f1, prec_at_k

def load_encoded_dataset(name):
    """Загружает X и y из папки final_data_encoded"""
    X_path = f"final_data_encoded/{name.replace(' ', '_')}_X.npy"
    y_path = f"final_data_encoded/{name.replace(' ', '_')}_y.npy"
    if not os.path.exists(X_path):
        return None, None
    X = np.load(X_path)
    y = np.load(y_path)
    return X, y

# Логистическая регрессия

In [6]:
from tqdm import tqdm

In [8]:
from sklearn.linear_model import LogisticRegression

C_VALUES = [0.01, 0.1, 1.0, 10.0, 100.0]

results_lr = []
for name in tqdm(DATASET_NAMES, desc="LR datasets"):
    X, y = load_encoded_dataset(name)
    if X is None:
        continue
    print(f"\nLR: {name} (X shape {X.shape})")

    auprc_list, auroc_list, f1_list, pk_list = [], [], [], []
    best_C_list = []

    for seed in tqdm(RANDOM_STATES, desc="Seeds", leave=False):
        X_tr, X_val, X_te, y_tr, y_val, y_te = split_stratified(X, y, TRAIN_SIZE, VAL_SIZE, TEST_SIZE, seed)
        # LR не требует подвыборки, так как сетка мала
        best_auprc = -1
        best_model = None
        best_c = None
        best_val_proba = None
        for C in C_VALUES:
            lr = LogisticRegression(C=C, class_weight='balanced', max_iter=1000, solver='lbfgs', random_state=seed)
            lr.fit(X_tr, y_tr)
            proba_val = lr.predict_proba(X_val)[:,1]
            prec, rec, _ = precision_recall_curve(y_val, proba_val)
            val_auprc = auc(rec, prec)
            if val_auprc > best_auprc:
                best_auprc = val_auprc
                best_model = lr
                best_c = C
                best_val_proba = proba_val
        proba_test = best_model.predict_proba(X_te)[:,1]
        best_thr = find_best_threshold(y_val, best_val_proba)
        auprc, auroc, f1, pk = compute_metrics(y_te, proba_test, best_thr)
        auprc_list.append(auprc); auroc_list.append(auroc); f1_list.append(f1); pk_list.append(pk)
        best_C_list.append(best_c)

    results_lr.append({
        "Dataset": name,
        "AUPRC_mean": np.mean(auprc_list), "AUPRC_std": np.std(auprc_list),
        "AUROC_mean": np.mean(auroc_list), "AUROC_std": np.std(auroc_list),
        "F1_mean": np.mean(f1_list), "F1_std": np.std(f1_list),
        "Prec@K_mean": np.nanmean(pk_list), "Prec@K_std": np.nanstd(pk_list),
        "Best_C": best_C_list[0]
    })

pd.DataFrame(results_lr).to_csv("lr_results_encoded.csv", index=False)
print("LR готово.")

LR datasets:   0%|          | 0/12 [00:00<?, ?it/s]


LR: Credit Card Fraud (X shape (284807, 30))



LR datasets:   8%|▊         | 1/12 [00:03<00:36,  3.33s/it]


LR: IEEE-CIS (X shape (590540, 367))



LR datasets:  17%|█▋        | 2/12 [27:09<2:39:39, 957.93s/it]

...


LR: Midterm-2018 (X shape (50538, 21))



LR datasets:  92%|█████████▏| 11/12 [32:43<00:28, 28.94s/it]


LR: MGTAB (X shape (10200, 788))



LR datasets: 100%|██████████| 12/12 [32:54<00:00, 164.55s/it]

LR готово.


In [10]:
from sklearn.ensemble import RandomForestClassifier

RF_PARAM_DIST = {
    'max_depth': [10, 20, 30, None],
    'min_samples_leaf': [1, 3, 5, 10],
    'max_features': ['sqrt', 'log2', None],
}
N_ITER_RF = 10

def tune_rf(X_train, y_train, X_val, y_val, seed):
    np.random.seed(seed)
    best_auprc = -1
    best_model = None
    best_params = None
    # Если данных много, используем подвыборку для подбора
    if len(X_train) > TUNE_SUBSAMPLE:
        idx = np.random.choice(len(X_train), TUNE_SUBSAMPLE, replace=False)
        X_tune = X_train[idx]
        y_tune = y_train[idx]
    else:
        X_tune, y_tune = X_train, y_train

    for _ in range(N_ITER_RF):
        params = {
            'max_depth': np.random.choice(RF_PARAM_DIST['max_depth']),
            'min_samples_leaf': np.random.choice(RF_PARAM_DIST['min_samples_leaf']),
            'max_features': np.random.choice(RF_PARAM_DIST['max_features']),
        }
        rf = RandomForestClassifier(n_estimators=500, class_weight='balanced_subsample', n_jobs=-1, random_state=seed, **params)
        rf.fit(X_tune, y_tune)
        proba_val = rf.predict_proba(X_val)[:,1]
        prec, rec, _ = precision_recall_curve(y_val, proba_val)
        val_auprc = auc(rec, prec)
        if val_auprc > best_auprc:
            best_auprc = val_auprc
            best_model = rf
            best_params = params
    # Финальная модель обучается на всех тренировочных данных (без подвыборки)
    final_rf = RandomForestClassifier(n_estimators=500, class_weight='balanced_subsample', n_jobs=-1, random_state=seed, **best_params)
    final_rf.fit(X_train, y_train)
    return final_rf, best_params

results_rf = []
for name in tqdm(DATASET_NAMES, desc="RF datasets"):
    X, y = load_encoded_dataset(name)
    if X is None:
        continue
    print(f"\nRF: {name} (X shape {X.shape})")

    auprc_list, auroc_list, f1_list, pk_list = [], [], [], []
    params_list = []
    for seed in tqdm(RANDOM_STATES, desc="Seeds", leave=False):
        X_tr, X_val, X_te, y_tr, y_val, y_te = split_stratified(X, y, TRAIN_SIZE, VAL_SIZE, TEST_SIZE, seed)
        model, best_params = tune_rf(X_tr, y_tr, X_val, y_val, seed)
        proba_test = model.predict_proba(X_te)[:,1]
        proba_val = model.predict_proba(X_val)[:,1]
        best_thr = find_best_threshold(y_val, proba_val)
        auprc, auroc, f1, pk = compute_metrics(y_te, proba_test, best_thr)
        auprc_list.append(auprc); auroc_list.append(auroc); f1_list.append(f1); pk_list.append(pk)
        params_list.append(best_params)

    results_rf.append({
        "Dataset": name,
        "AUPRC_mean": np.mean(auprc_list), "AUPRC_std": np.std(auprc_list),
        "AUROC_mean": np.mean(auroc_list), "AUROC_std": np.std(auroc_list),
        "F1_mean": np.mean(f1_list), "F1_std": np.std(f1_list),
        "Prec@K_mean": np.nanmean(pk_list), "Prec@K_std": np.nanstd(pk_list),
        "Best_params": params_list[0]
    })

pd.DataFrame(results_rf).to_csv("rf_results_encoded.csv", index=False)
print("RF готово.")

RF datasets:   0%|          | 0/12 [00:00<?, ?it/s]


RF: Credit Card Fraud (X shape (284807, 30))



RF datasets:   8%|▊         | 1/12 [11:52<2:10:33, 712.16s/it]


RF: IEEE-CIS (X shape (590540, 367))



RF datasets:  17%|█▋        | 2/12 [1:44:13<9:52:10, 3553.05s/it]

...


RF: Midterm-2018 (X shape (50538, 21))



RF datasets:  92%|█████████▏| 11/12 [3:15:07<05:40, 340.09s/it]


RF: MGTAB (X shape (10200, 788))



RF datasets: 100%|██████████| 12/12 [4:23:03<00:00, 1315.28s/it]

RF готово.


In [7]:
import xgboost as xgb
from tqdm import tqdm

XGB_PARAM_DIST = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'n_estimators': [200, 400, 600, 800],
    'max_depth': [3, 5, 7, 9],
    'reg_lambda': [0.1, 1, 10],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
}
N_ITER_XGB = 10

def tune_xgb(X_train, y_train, X_val, y_val, seed):
    np.random.seed(seed)
    best_auprc = -1
    best_model = None
    best_params = None
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    if len(X_train) > TUNE_SUBSAMPLE:
        idx = np.random.choice(len(X_train), TUNE_SUBSAMPLE, replace=False)
        X_tune = X_train[idx]
        y_tune = y_train[idx]
    else:
        X_tune, y_tune = X_train, y_train

    for _ in range(N_ITER_XGB):
        params = {
            'learning_rate': np.random.choice(XGB_PARAM_DIST['learning_rate']),
            'n_estimators': np.random.choice(XGB_PARAM_DIST['n_estimators']),
            'max_depth': np.random.choice(XGB_PARAM_DIST['max_depth']),
            'reg_lambda': np.random.choice(XGB_PARAM_DIST['reg_lambda']),
            'subsample': np.random.choice(XGB_PARAM_DIST['subsample']),
            'colsample_bytree': np.random.choice(XGB_PARAM_DIST['colsample_bytree']),
            'objective': 'binary:logistic',
            'eval_metric': 'aucpr',
            'tree_method': 'hist',
            'scale_pos_weight': scale_pos_weight,
            'early_stopping_rounds': 20,
            'seed': seed,
            'verbosity': 0,
        }
        model = xgb.XGBClassifier(**params)
        model.fit(X_tune, y_tune, eval_set=[(X_val, y_val)], verbose=False)
        proba_val = model.predict_proba(X_val)[:,1]
        prec, rec, _ = precision_recall_curve(y_val, proba_val)
        val_auprc = auc(rec, prec)
        if val_auprc > best_auprc:
            best_auprc = val_auprc
            best_model = model
            best_params = params
    # Финальная модель на всех данных
    final_params = best_params.copy()
    final_model = xgb.XGBClassifier(**final_params)
    final_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return final_model, best_params

results_xgb = []
for name in tqdm(DATASET_NAMES, desc="XGB datasets"):
    X, y = load_encoded_dataset(name)
    if X is None:
        continue
    print(f"\nXGB: {name} (X shape {X.shape})")

    auprc_list, auroc_list, f1_list, pk_list = [], [], [], []
    params_list = []
    for seed in tqdm(RANDOM_STATES, desc="Seeds", leave=False):
        X_tr, X_val, X_te, y_tr, y_val, y_te = split_stratified(X, y, TRAIN_SIZE, VAL_SIZE, TEST_SIZE, seed)
        model, best_params = tune_xgb(X_tr, y_tr, X_val, y_val, seed)
        proba_test = model.predict_proba(X_te)[:,1]
        proba_val = model.predict_proba(X_val)[:,1]
        best_thr = find_best_threshold(y_val, proba_val)
        auprc, auroc, f1, pk = compute_metrics(y_te, proba_test, best_thr)
        auprc_list.append(auprc); auroc_list.append(auroc); f1_list.append(f1); pk_list.append(pk)
        params_list.append(best_params)

    results_xgb.append({
        "Dataset": name,
        "AUPRC_mean": np.mean(auprc_list), "AUPRC_std": np.std(auprc_list),
        "AUROC_mean": np.mean(auroc_list), "AUROC_std": np.std(auroc_list),
        "F1_mean": np.mean(f1_list), "F1_std": np.std(f1_list),
        "Prec@K_mean": np.nanmean(pk_list), "Prec@K_std": np.nanstd(pk_list),
        "Best_params": params_list[0]
    })

pd.DataFrame(results_xgb).to_csv("xgb_results_encoded.csv", index=False)
print("XGB готово.")

XGB datasets:   0%|          | 0/12 [00:00<?, ?it/s]


XGB: Credit Card Fraud (X shape (284807, 30))



XGB datasets:   8%|▊         | 1/12 [00:23<04:16, 23.33s/it]


XGB: IEEE-CIS (X shape (590540, 367))



XGB datasets:  17%|█▋        | 2/12 [15:49<1:32:24, 554.50s/it]

...


XGB: Midterm-2018 (X shape (50538, 21))



XGB datasets:  92%|█████████▏| 11/12 [32:50<00:44, 44.16s/it]


XGB: MGTAB (X shape (10200, 788))



XGB datasets: 100%|██████████| 12/12 [35:47<00:00, 178.99s/it]

XGB готово.


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims=[256,128,64], dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for hdim in hidden_dims:
            layers.append(nn.Linear(prev, hdim))
            layers.append(nn.BatchNorm1d(hdim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = hdim
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

def train_mlp(model, X_train, y_train, X_val, y_val, device='cpu', batch_size=512, max_epochs=100, patience=15, lr=0.001, weight_decay=1e-5):
    model.to(device)

    if len(X_train) > TUNE_SUBSAMPLE:
        X_train, _, y_train, _ = train_test_split(
            X_train, y_train, train_size=TUNE_SUBSAMPLE, stratify=y_train, random_state=seed
        )

    pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
    pos_weight = torch.tensor(pos_weight, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=7)

    X_train_t = torch.tensor(X_train.astype(np.float32), device=device)
    y_train_t = torch.tensor(y_train.astype(np.float32), device=device).view(-1,1)
    X_val_t = torch.tensor(X_val.astype(np.float32), device=device)
    y_val_t = torch.tensor(y_val.astype(np.float32), device=device).view(-1,1)

    train_dataset = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    best_val_auprc = 0
    best_state = None
    wait = 0
    for epoch in range(max_epochs):
        model.train()
        for Xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(Xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_proba = torch.sigmoid(val_logits).cpu().numpy().ravel()
            prec, rec, _ = precision_recall_curve(y_val, val_proba)
            val_auprc = auc(rec, prec)
        scheduler.step(val_auprc)
        if val_auprc > best_val_auprc:
            best_val_auprc = val_auprc
            best_state = model.state_dict().copy()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

results_mlp = []
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for name in tqdm(DATASET_NAMES, desc="MLP datasets"):
    X, y = load_encoded_dataset(name)
    if X is None:
        continue
    print(f"\nMLP: {name} (X shape {X.shape})")

    auprc_list, auroc_list, f1_list, pk_list = [], [], [], []
    for seed in tqdm(RANDOM_STATES, desc="Seeds", leave=False):
        X_tr, X_val, X_te, y_tr, y_val, y_te = split_stratified(X, y, TRAIN_SIZE, VAL_SIZE, TEST_SIZE, seed)
        model = MLPClassifier(input_dim=X_tr.shape[1])
        model = train_mlp(model, X_tr, y_tr, X_val, y_val, device=device)
        X_te_t = torch.tensor(X_te.astype(np.float32), device=device)
        model.eval()
        with torch.no_grad():
            proba_test = torch.sigmoid(model(X_te_t)).cpu().numpy().ravel()
        X_val_t = torch.tensor(X_val.astype(np.float32), device=device)
        with torch.no_grad():
            proba_val = torch.sigmoid(model(X_val_t)).cpu().numpy().ravel()
        best_thr = find_best_threshold(y_val, proba_val)
        auprc, auroc, f1, pk = compute_metrics(y_te, proba_test, best_thr)
        auprc_list.append(auprc); auroc_list.append(auroc); f1_list.append(f1); pk_list.append(pk)

    results_mlp.append({
        "Dataset": name,
        "AUPRC_mean": np.mean(auprc_list), "AUPRC_std": np.std(auprc_list),
        "AUROC_mean": np.mean(auroc_list), "AUROC_std": np.std(auroc_list),
        "F1_mean": np.mean(f1_list), "F1_std": np.std(f1_list),
        "Prec@K_mean": np.nanmean(pk_list), "Prec@K_std": np.nanstd(pk_list),
    })

pd.DataFrame(results_mlp).to_csv("mlp_results_encoded.csv", index=False)
print("MLP готово.")

MLP datasets:   0%|          | 0/12 [00:00<?, ?it/s]


MLP: Credit Card Fraud (X shape (284807, 30))



MLP datasets:   8%|▊         | 1/12 [00:32<05:53, 32.12s/it]


MLP: IEEE-CIS (X shape (590540, 367))



MLP datasets:  17%|█▋        | 2/12 [01:58<10:38, 63.82s/it]

...


MLP: Midterm-2018 (X shape (50538, 21))



MLP datasets:  92%|█████████▏| 11/12 [11:34<00:50, 50.59s/it]


MLP: MGTAB (X shape (10200, 788))



MLP datasets: 100%|██████████| 12/12 [11:42<00:00, 58.53s/it]

MLP готово.


In [8]:
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=32, hidden_dims=[128,64]):
        super(VAE, self).__init__()
        self.latent_dim = latent_dim
        encoder = []
        prev = input_dim
        for hdim in hidden_dims:
            encoder.append(nn.Linear(prev, hdim))
            encoder.append(nn.ReLU())
            prev = hdim
        self.encoder = nn.Sequential(*encoder)
        self.fc_mu = nn.Linear(prev, latent_dim)
        self.fc_logvar = nn.Linear(prev, latent_dim)
        decoder = []
        prev = latent_dim
        for hdim in reversed(hidden_dims):
            decoder.append(nn.Linear(prev, hdim))
            decoder.append(nn.ReLU())
            prev = hdim
        decoder.append(nn.Linear(prev, input_dim))
        self.decoder = nn.Sequential(*decoder)

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

def vae_loss(recon, x, mu, logvar, beta=0.5):
    recon_loss = nn.functional.mse_loss(recon, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl_loss

def train_vae(model, X_train_norm, X_val_norm, device='cpu', batch_size=256, max_epochs=80, patience=10, lr=0.001, beta=0.5):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Если данных много, используем подвыборку для обучения
    if len(X_train_norm) > TUNE_SUBSAMPLE:
        idx = np.random.choice(len(X_train_norm), TUNE_SUBSAMPLE, replace=False)
        X_train_norm = X_train_norm[idx]
    if len(X_val_norm) > TUNE_SUBSAMPLE:
        idx = np.random.choice(len(X_val_norm), TUNE_SUBSAMPLE, replace=False)
        X_val_norm = X_val_norm[idx]

    X_train_t = torch.tensor(X_train_norm.astype(np.float32), device=device)
    X_val_t = torch.tensor(X_val_norm.astype(np.float32), device=device)
    train_dataset = TensorDataset(X_train_t)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    best_val_loss = np.inf
    best_state = None
    wait = 0
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0
        for (xb,) in train_loader:
            optimizer.zero_grad()
            recon, mu, logvar = model(xb)
            loss = vae_loss(recon, xb, mu, logvar, beta)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        model.eval()
        with torch.no_grad():
            recon, mu, logvar = model(X_val_t)
            val_loss = vae_loss(recon, X_val_t, mu, logvar, beta).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict().copy()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def anomaly_score(model, X, beta=0.5, device='cpu'):
    model.eval()
    X_t = torch.tensor(X.astype(np.float32), device=device)
    with torch.no_grad():
        recon, mu, logvar = model(X_t)
        recon_loss = nn.functional.mse_loss(recon, X_t, reduction='none').sum(dim=1)
        kl_loss = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(dim=1)
        scores = (recon_loss + beta * kl_loss).cpu().numpy()
        scores = np.nan_to_num(scores, nan=1e10, posinf=1e10, neginf=-1e10)
    return scores

results_vae = []
for name in tqdm(DATASET_NAMES, desc="VAE datasets"):
    X, y = load_encoded_dataset(name)
    if X is None:
        continue
    print(f"\nVAE: {name} (X shape {X.shape})")

    auprc_list, auroc_list, pk_list = [], [], []
    for seed in tqdm(RANDOM_STATES, desc="Seeds", leave=False):
        X_tr, X_val, X_te, y_tr, y_val, y_te = split_stratified(X, y, TRAIN_SIZE, VAL_SIZE, TEST_SIZE, seed)
        # Обучаем только на нормальных (y=0)
        X_tr_norm = X_tr[y_tr == 0]
        X_val_norm = X_val[y_val == 0]
        if len(X_tr_norm) == 0 or len(X_val_norm) == 0:
            print(f"    seed {seed}: нет нормальных объектов, пропуск")
            continue
        model = VAE(input_dim=X_tr.shape[1])
        model = train_vae(model, X_tr_norm, X_val_norm, device=device)
        scores = anomaly_score(model, X_te, device=device)
        # Нормируем в [0,1]
        proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
        proba = np.nan_to_num(proba, nan=0.0)
        auprc, auroc, _, _ = compute_metrics(y_te, proba, 0.5)  # порог не важен для AUPRC/AUROC
        K = y_te.sum()
        if K > 0:
            top_k_idx = np.argsort(proba)[-K:]
            prec_at_k = y_te[top_k_idx].mean()
        else:
            prec_at_k = np.nan
        auprc_list.append(auprc); auroc_list.append(auroc); pk_list.append(prec_at_k)

    results_vae.append({
        "Dataset": name,
        "AUPRC_mean": np.mean(auprc_list), "AUPRC_std": np.std(auprc_list),
        "AUROC_mean": np.mean(auroc_list), "AUROC_std": np.std(auroc_list),
        "Prec@K_mean": np.nanmean(pk_list), "Prec@K_std": np.nanstd(pk_list),
    })

pd.DataFrame(results_vae).to_csv("vae_results_encoded.csv", index=False)
print("VAE готово.")

VAE datasets:   0%|          | 0/12 [00:00<?, ?it/s]


VAE: Credit Card Fraud (X shape (284807, 30))



VAE datasets:   8%|▊         | 1/12 [00:56<10:20, 56.39s/it]


VAE: IEEE-CIS (X shape (590540, 367))



VAE datasets:  17%|█▋        | 2/12 [02:03<10:27, 62.73s/it]

...


VAE: Midterm-2018 (X shape (50538, 21))



VAE datasets:  92%|█████████▏| 11/12 [05:35<00:15, 15.69s/it]


VAE: MGTAB (X shape (10200, 788))



VAE datasets: 100%|██████████| 12/12 [05:51<00:00, 29.32s/it]

VAE готово.
